In [0]:
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

Add secret to Databricks Scope

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.put_secret("aws-rds-practice-scope","username",string_value ="usaha1")
w.secrets.put_secret("aws-rds-practice-scope","password",string_value ="C0pacabana")
w.secrets.put_secret("aws-rds-practice-scope","host",string_value ="practicedb.cafom8oi2mlb.us-east-1.rds.amazonaws.com")
w.secrets.put_secret("aws-rds-practice-scope","database",string_value ="postgres")

Extract secret from Scope

In [0]:
user = dbutils.secrets.get(scope = "aws-rds-practice-scope", key = "username")
psw = dbutils.secrets.get(scope = "aws-rds-practice-scope", key = "password")
host = dbutils.secrets.get(scope = "aws-rds-practice-scope", key = "host")
database = dbutils.secrets.get(scope = "aws-rds-practice-scope", key = "database")

Read data from external database table

In [0]:
sales_raw_df = spark.read \
    .format("postgresql") \
    .option("host", host) \
    .option("port", "5432") \
    .option("database", database) \
    .option("dbtable", "sales.sales_data") \
    .option("user", user) \
    .option("password", psw) \
    .option("customSchema", file_schema) \
    .load()

In [0]:
display(sales_raw_df)

In [0]:
sales_raw_df.describe().display()

<b>Problems identified from analysis which need to be fixed:</b>

1. Convert id from integer to string and rename it as transaction_id.
2. Rename the name column to customer_name.
3. Convert the dop to date format and rename the column to date_of_purchase.
4. Rename the phone column to customer_phone
5. Convert the amount to a long value and filter out nulls and outlier values.
6. Rename the column to purchase_amount
7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

In [0]:
from pyspark.sql.functions import expr

sales_df = sales_raw_df.selectExpr(
    "cast(id AS STRING) as transaction_id",
    "name as customer_name",
    "nvl(try_cast(dop as date), to_date(dop, 'dd-MM-yyyy')) as date_of_purchase",
    "phone as customer_phone",
    "cast(nvl(amount, 0) as long) as purchase_amount",
    "nvl(try_cast(discount as double), 0) as applied_discount"
)

In [0]:
sales_df.display()

In [0]:
sales_df.describe().display()